# 🔧 段階2: 環境構築とハンズオン実装
**実際にコードを動かしながら学ぶ - Python 実装ガイド**

---

## 📚 目次
1. [環境構築](#1-環境構築)
2. [プロジェクトセットアップ](#2-プロジェクトセットアップ)
3. [最初の推論：Hello LLM](#3-最初の推論-hello-llm)
4. [Tokenizer を理解する](#4-tokenizer-を理解する)
5. [埋め込み（Embedding）を実装](#5-embedding-を実装)
6. [ハイパーパラメータの影響](#6-ハイパーパラメータの影響)
7. [実践課題](#7-実践課題)

## 1. 環境構築

### Step 1: インストール確認

In [ ]:
import sys
print(f"Python: {sys.version}")

import torch
import transformers
import numpy as np
import pandas as pd

print(f"PyTorch バージョン   : {torch.__version__}")
print(f"Transformers バージョン: {transformers.__version__}")
print(f"GPU 利用可能         : {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用デバイス         : {device}")

## 2. プロジェクトセットアップ

### プロジェクト構造確認

In [ ]:
from pathlib import Path

project_root = Path.cwd().parent
print(f"プロジェクトルート: {project_root}")

dirs_to_check = ["src", "embeddings", "fine_tuned_model", "checkpoints", "corpus", "docs"]
for d in dirs_to_check:
    path = project_root / d
    status = "✅" if path.exists() else "❌"
    print(f"  {status} {d}/")

## 3. 最初の推論 Hello LLM

### 例1: 感情分類（distilbert）

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
)

texts = [
    "This is wonderful!",
    "This is terrible.",
    "I really like this.",
]

results = classifier(texts)
for text, result in zip(texts, results):
    label = result["label"]
    score = result["score"]
    bar = "▓" * int(score * 20)
    print(f"{label:8s} {score:.4f} [{bar:<20}]  {text}")

### 例2: テキスト生成（GPT-2）

In [ ]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer_gpt2 = GPT2Tokenizer.from_pretrained("gpt2")
model_gpt2 = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
tokenizer_gpt2.pad_token = tokenizer_gpt2.eos_token

def generate_text(prompt: str, max_new_tokens: int = 40, temperature: float = 0.7) -> str:
    input_ids = tokenizer_gpt2.encode(prompt, return_tensors="pt").to(device)
    output = model_gpt2.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer_gpt2.eos_token_id,
    )
    return tokenizer_gpt2.decode(output[0], skip_special_tokens=True)

prompt = "Artificial intelligence is"
for temp in [0.7, 0.8, 0.9]:
    print(f"\n--- temperature={temp} ---")
    print(generate_text(prompt, temperature=temp))

## 4. Tokenizer を理解する

Tokenizer の役割: テキスト → トークン → 数値ID

In [ ]:
from transformers import BertTokenizer

tokenizer_bert = BertTokenizer.from_pretrained("bert-base-uncased")

texts = [
    "Hello, how are you?",
    "The quick brown fox jumps.",
    "AI is amazing!",
]

for text in texts:
    tokens    = tokenizer_bert.tokenize(text)
    token_ids = tokenizer_bert.convert_tokens_to_ids(tokens)
    encoded   = tokenizer_bert.encode(text, add_special_tokens=True)
    print(f"\nOriginal : {text}")
    print(f"Tokens   : {tokens}")
    print(f"IDs      : {token_ids}")
    print(f"Encoded  : {encoded}  ← [CLS]=101, [SEP]=102")

### パディングと截断（バッチ処理）

In [ ]:
from transformers import AutoTokenizer

tokenizer_auto = AutoTokenizer.from_pretrained("bert-base-uncased")

batch_texts = [
    "Short text.",
    "This is a medium length text that is a bit longer.",
    "This is a very long text. " * 8,
]

encoded = tokenizer_auto(
    batch_texts,
    padding=True,
    truncation=True,
    max_length=64,
    return_tensors="pt",
)

print(f"input_ids shape    : {encoded['input_ids'].shape}")
print(f"attention_mask shape: {encoded['attention_mask'].shape}")
print("\n-- attention_mask (1=実トークン, 0=パディング) --")
for i, mask in enumerate(encoded["attention_mask"]):
    real_len = mask.sum().item()
    print(f"  文{i+1}: 実{real_len:2d}トークン  {mask[:20].tolist()}...")

## 5. Embedding を実装

Embedding = テキストの「意味」を数値ベクトルで表現

In [ ]:
import torch
import numpy as np
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity

tokenizer_emb = BertTokenizer.from_pretrained("bert-base-uncased")
model_emb = BertModel.from_pretrained("bert-base-uncased")
model_emb.eval()

def get_embeddings(texts: list) -> np.ndarray:
    enc = tokenizer_emb(texts, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        out = model_emb(**enc)
    return out.last_hidden_state[:, 0, :].numpy()

test_sentences = [
    "The cat is sleeping",
    "A dog is playing",
    "The weather is nice today",
]

embeddings = get_embeddings(test_sentences)
print(f"Embedding shape: {embeddings.shape}  (文数 × 次元数)\n")

sim_matrix = cosine_similarity(embeddings)
print("コサイン類似度:")
for i in range(len(test_sentences)):
    for j in range(i + 1, len(test_sentences)):
        print(f"  [{test_sentences[i]:30s}]")
        print(f"  [{test_sentences[j]:30s}]")
        print(f"  → 類似度: {sim_matrix[i][j]:.4f}\n")

In [ ]:
import matplotlib.pyplot as plt

labels = [s[:20] for s in test_sentences]
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(sim_matrix, vmin=0, vmax=1, cmap="Blues")
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=8)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center", fontsize=10)
plt.colorbar(im, ax=ax, label="Cosine Similarity")
ax.set_title("文の類似度ヒートマップ")
plt.tight_layout()
plt.show()

## 6. ハイパーパラメータの影響

学習率（learning rate）を変えると損失の下がり方がどう変わるか確認します。

In [ ]:
import torch
from torch import nn
from torch.optim import Adam
import matplotlib.pyplot as plt

def train_with_lr(learning_rate: float, epochs: int = 20, seed: int = 42) -> list:
    torch.manual_seed(seed)
    X = torch.randn(200, 10)
    y = torch.randint(0, 2, (200,))
    model = nn.Sequential(nn.Linear(10, 64), nn.ReLU(), nn.Linear(64, 2))
    optimizer = Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.CrossEntropyLoss()
    losses = []
    for _ in range(epochs):
        out = model(X)
        loss = loss_fn(out, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

learning_rates = [1e-4, 1e-3, 1e-2, 1e-1]
epochs = 20

plt.figure(figsize=(8, 4))
print(f"{'lr':8}  {'初期損失':>8}  {'最終損失':>8}")
print("-" * 35)
for lr in learning_rates:
    losses = train_with_lr(lr, epochs=epochs)
    has_nan = any(v != v for v in losses)
    final = losses[-1] if not has_nan else float("nan")
    print(f"{lr:.4f}    {losses[0]:8.4f}    {final:8.4f}" + ("  ← 発散" if has_nan else ""))
    if not has_nan:
        plt.plot(range(1, epochs + 1), losses, label=f"lr={lr}")

plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("学習率による損失の変化")
plt.legend(); plt.tight_layout(); plt.show()

## 7. 実践課題

### 課題1: 感情分析バッチ処理

In [ ]:
def analyze_sentiment_batch(texts: list) -> list:
    from transformers import pipeline
    clf = pipeline("sentiment-analysis")
    return clf(texts)

tweets = [
    "I love this product!",
    "Worst purchase ever.",
    "It's okay, nothing special.",
    "Absolutely fantastic experience!",
]

results = analyze_sentiment_batch(tweets)
print(f"{'Label':10} {'Score':6}  Text")
print("-" * 60)
for tweet, r in zip(tweets, results):
    print(f"{r['label']:10} {r['score']:.4f}  {tweet}")

### 課題2: 2文のコサイン類似度を計算

In [ ]:
def compute_similarity(text1: str, text2: str) -> float:
    from sklearn.metrics.pairwise import cosine_similarity
    embs = get_embeddings([text1, text2])
    return float(cosine_similarity(embs)[0][1])

pairs = [
    ("The cat is on the mat", "A cat is sitting on a mat"),
    ("I love Python", "I hate Mondays"),
    ("Machine learning is hard", "Deep learning requires data"),
]

print(f"{'類似度':6}  テキストペア")
print("-" * 70)
for t1, t2 in pairs:
    sim = compute_similarity(t1, t2)
    print(f"  {sim:.4f}  [{t1}]  ←→  [{t2}]")

## ✅ 理解度チェック

- [ ] `torch.cuda.is_available()` の意味が説明できる  
- [ ] CLS トークンが文全体の表現に使われる理由が分かる  
- [ ] 学習率が大きすぎると損失が発散することを確認した  
- [ ] コサイン類似度 1.0 が「完全一致」を意味することが分かる  
- [ ] 課題1・2 を自分のテキストで試してみた  

---

✅ 完了したら → **[段階3: 実践応用編](04_advanced_implementation.md)** へ